# AgentCore Gateway Demo — Infrastructure Setup (Open Insurance)

This notebook sets up the infrastructure for demonstrating **centralized MCP server management** with **auth propagation** using AWS AgentCore Gateway — themed for an insurance use case.

## Architecture

```
 User (CLI)
   │
   │ username + password (Resource Owner Password grant)
   ▼
 Okta Token Endpoint ──▶ JWT (with groups, client_id claims)
   │
   ▼
 Strands Agent (JWT attached as Bearer token)
   │
   │ MCP (StreamableHTTP + Bearer JWT)
   ▼
 AgentCore Gateway
   ├── Ingress Auth: validates JWT (signature, audience, client_id)
   ├── Cedar Policy Engine (ENFORCE mode)
   │     • principal is AgentCore::OAuthUser::"<JWT sub>"
   │     • action == AgentCore::Action::"<TargetName>"
   │     • resource == AgentCore::Gateway::"<gateway-arn>"
   │     • when { context.input constraints }  ← contextual control
   │
   ├───────────────┐
   ▼               ▼
 Lambda         Lambda
 (PolicyLookup) (ClaimsData)
```

## What this notebook does

1. Install dependencies and load Okta config
2. Deploy **all** infrastructure via Terraform (Lambda, IAM)
3. Verify Lambda functions
4. Create AgentCore Gateway with Okta OIDC auth
5. Create Policy Engine + Cedar policies (ENFORCE mode)
6. Save config for demo notebooks

## Insurance Demo Personas

| User | Role | Okta Group | PolicyLookup | ClaimsData |
|------|------|-----------|-------------|------------|
| Alice | Customer Service Rep | `analysts` | ALLOWED | BLOCKED |
| Bob | Claims Adjuster | `finance-admins` | ALLOWED | ALLOWED (full) |
| Contextual user (`CONTEXTUAL_USER`) | Demo User | — | ALLOWED | ALLOWED only when `max_amount <= CONTEXTUAL_MAX_AMOUNT` (Cedar contextual) |

> **Note:** Demo users (Alice, Bob) are created by `00_okta_setup.ipynb`. The contextual demo user is configured via the `CONTEXTUAL_USER` and `CONTEXTUAL_MAX_AMOUNT` environment variables in `.env`.

## Cell 1: Dependencies + Configuration

Before running this notebook:
1. Run `00_okta_setup.ipynb` to configure the Okta authorization server (scopes, claims, access policy) and create demo users (Alice, Bob)
2. Copy `.env.example` to `.env` and fill in your values

Demo notebooks (`02_claude_code_setup`, `05_strands_demo`) create their own OIDC apps and register them with the Gateway.

In [ ]:
%pip install -q boto3 strands-agents strands-agents-tools mcp httpx python-dotenv PyJWT requests

import os
import json
import time
import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

# --- Okta Configuration ---
OKTA_DOMAIN = os.environ["OKTA_DOMAIN"]           # e.g., "dev-12345678.okta.com"
OKTA_ISSUER = f"https://{OKTA_DOMAIN}/oauth2/default"
OKTA_JWKS_URI = f"{OKTA_ISSUER}/v1/keys"

# --- Demo users (created by 00_okta_setup.ipynb, stored in .env) ---
BOB_USERNAME = os.environ.get("BOB_USERNAME", "")
ALICE_USERNAME = os.environ.get("ALICE_USERNAME", "")

# --- SPA client ID (created by 00_okta_setup.ipynb, used as initial allowedClient) ---
OKTA_SPA_CLIENT_ID = os.environ.get("OKTA_SPA_CLIENT_ID", "")

# --- AWS Configuration ---
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")
ACCOUNT_ID = boto3.client("sts", region_name=AWS_REGION).get_caller_identity()["Account"]

# --- Demo naming ---
DEMO_PREFIX = "agentcore-mcp-demo"

# --- Contextual demo user (Cedar input constraint demo) ---
CONTEXTUAL_USER = os.environ.get("CONTEXTUAL_USER", "")
CONTEXTUAL_MAX_AMOUNT = int(os.environ.get("CONTEXTUAL_MAX_AMOUNT", "100000"))

print(f"Okta Domain:    {OKTA_DOMAIN}")
print(f"Okta Issuer:    {OKTA_ISSUER}")
print(f"AWS Region:     {AWS_REGION}")
print(f"AWS Account:    {ACCOUNT_ID}")
print(f"Demo Prefix:    {DEMO_PREFIX}")
if OKTA_SPA_CLIENT_ID:
    print(f"SPA Client ID:  {OKTA_SPA_CLIENT_ID}")
if ALICE_USERNAME:
    print(f"Alice:          {ALICE_USERNAME}")
if BOB_USERNAME:
    print(f"Bob:            {BOB_USERNAME}")
if CONTEXTUAL_USER:
    print(f"Contextual User: {CONTEXTUAL_USER} (max_amount <= {CONTEXTUAL_MAX_AMOUNT})")
else:
    print(f"Contextual User: (not set — set CONTEXTUAL_USER in .env for Cedar contextual demo)")
if not OKTA_SPA_CLIENT_ID:
    print(f"\n⚠️  OKTA_SPA_CLIENT_ID not set — run 00_okta_setup.ipynb first to create the SPA app")
if not BOB_USERNAME:
    print(f"\n⚠️  BOB_USERNAME not set — run 00_okta_setup.ipynb first to create demo users")
print("\n✅ Configuration loaded successfully")

## Cell 2: Deploy All Infrastructure (Terraform)

This runs `terraform apply` to create **all** AWS infrastructure:
- **Lambda** functions (PolicyLookup + ClaimsData) with inline Python code
- **IAM** roles for Lambda execution and Gateway

After this cell, we verify Lambdas (Cell 3).

In [ ]:
import subprocess

TERRAFORM_DIR = os.path.join(os.getcwd(), "terraform")

# Common Terraform variables (passed to all terraform commands)
TF_VARS = [
    f"-var=aws_region={AWS_REGION}",
    f"-var=demo_prefix={DEMO_PREFIX}",
    f"-var=contextual_user={CONTEXTUAL_USER}",
    f"-var=contextual_max_amount={CONTEXTUAL_MAX_AMOUNT}",
]

# --- Step 1: Terraform init ---
print("Initializing Terraform...")
subprocess.run(["terraform", "init", "-upgrade"], cwd=TERRAFORM_DIR, check=True)

# --- Step 2: Full Terraform apply (Lambda, IAM) ---
print("\nApplying full Terraform configuration...")
subprocess.run(
    ["terraform", "apply", "-auto-approve", *TF_VARS],
    cwd=TERRAFORM_DIR, check=True,
)

# --- Step 3: Capture all Terraform outputs ---
tf_outputs = {}
for key in ["policy_lookup_lambda_arn", "policy_lookup_lambda_name",
            "claims_lambda_arn", "claims_lambda_name", "lambda_role_name",
            "gateway_role_arn"]:
    r = subprocess.run(
        ["terraform", "output", "-raw", key],
        cwd=TERRAFORM_DIR, capture_output=True, text=True, check=True,
    )
    tf_outputs[key] = r.stdout.strip()

POLICY_LOOKUP_FUNCTION_NAME = tf_outputs["policy_lookup_lambda_name"]
CLAIMS_FUNCTION_NAME = tf_outputs["claims_lambda_name"]
POLICY_LOOKUP_LAMBDA_ARN = tf_outputs["policy_lookup_lambda_arn"]
CLAIMS_LAMBDA_ARN = tf_outputs["claims_lambda_arn"]
LAMBDA_ROLE_NAME = tf_outputs["lambda_role_name"]
GATEWAY_ROLE_ARN = tf_outputs["gateway_role_arn"]

print(f"\n--- Terraform Outputs ---")
print(f"PolicyLookup Lambda:  {POLICY_LOOKUP_FUNCTION_NAME}")
print(f"ClaimsData Lambda:    {CLAIMS_FUNCTION_NAME}")
print(f"Lambda role:          {LAMBDA_ROLE_NAME}")
print(f"Gateway role:         {GATEWAY_ROLE_ARN}")
if CONTEXTUAL_USER:
    print(f"Contextual user:      {CONTEXTUAL_USER} (max_amount <= {CONTEXTUAL_MAX_AMOUNT})")
print(f"\n✅ All infrastructure created via Terraform")

## Cell 3: Verify Lambda Functions

Invokes both Lambda functions directly to confirm they were deployed correctly by Terraform.
- **PolicyLookup**: Returns insurance policy details (accessible by all users)
- **ClaimsData**: Returns confidential claims information (restricted to claims adjusters)

In [ ]:
lambda_client = boto3.client("lambda", region_name=AWS_REGION)

# --- Verify PolicyLookup Lambda ---
response = lambda_client.invoke(
    FunctionName=POLICY_LOOKUP_FUNCTION_NAME,
    Payload=json.dumps({"policy_number": "POL-10042"})
)
result = json.loads(response["Payload"].read())
print(f"✅ PolicyLookup Lambda test: {json.dumps(json.loads(result['body']), indent=2)}")

# --- Verify ClaimsData Lambda ---
response = lambda_client.invoke(
    FunctionName=CLAIMS_FUNCTION_NAME,
    Payload=json.dumps({"query": "claims summary"})
)
result = json.loads(response["Payload"].read())
print(f"\n✅ ClaimsData Lambda test: {json.dumps(json.loads(result['body']), indent=2)}")

## Cell 4: Create AgentCore Gateway with Okta OIDC Auth

This creates the AgentCore Gateway with:
- **Okta as the direct OIDC provider** (no Cognito intermediary)
- **2 targets**: PolicyLookup Lambda, ClaimsData Lambda
- **Ingress auth**: Gateway validates Okta JWTs against the JWKS endpoint

**Insurance Demo Targets:**
| Target | Tool | Access | Description |
|--------|------|--------|-------------|
| PolicyLookup | `lookup_policy` | All users | Look up insurance policy details by policy number |
| ClaimsData | `query_claims` | Claims adjusters only | Query confidential claims data |

**Why `roleArn`?** The Gateway is a managed AWS service that acts on your behalf — like a Lambda function, it needs an IAM execution role. The `roleArn` parameter assigns the Gateway an IAM role so it can:
1. **Invoke Lambda targets** — when users call tools like `lookup_policy`, the Gateway makes the `lambda:InvokeFunction` call using this role
2. **Evaluate Cedar policies** — the policy engine attachment (Cell 5) requires `bedrock-agentcore:*` permissions on this role

Without `roleArn`, the Gateway has no AWS identity and cannot reach any of your backends. The role is created by Terraform in Cell 2 (see `gateway_role_arn` output) with a trust policy allowing the AgentCore service to assume it.

In [ ]:
# --- AgentCore Gateway control plane client ---
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=AWS_REGION)

GATEWAY_NAME = f"{DEMO_PREFIX}-gateway"
OKTA_DISCOVERY_URL = f"{OKTA_ISSUER}/.well-known/openid-configuration"

# --- Step 0: Ensure Okta auth server has a 'client_id' claim ---
# AgentCore Gateway checks allowedClients against the 'client_id' claim in the JWT.
# Okta puts the client ID in the 'cid' claim by default, so we add a custom 'client_id'
# claim that maps to the app's client ID.
print("Configuring Okta auth server 'client_id' claim...")
import requests as http_requests
OKTA_API_TOKEN = os.environ.get("OKTA_API_TOKEN", "")
if OKTA_API_TOKEN:
    resp = http_requests.post(
        f"{OKTA_ISSUER.replace('/oauth2/default', '')}/api/v1/authorizationServers/default/claims",
        headers={
            "Authorization": f"SSWS {OKTA_API_TOKEN}",
            "Content-Type": "application/json",
        },
        json={
            "name": "client_id",
            "status": "ACTIVE",
            "claimType": "RESOURCE",
            "valueType": "EXPRESSION",
            "value": "app.clientId",
            "alwaysIncludeInToken": True,
            "conditions": {"scopes": []},
        },
    )
    if resp.status_code == 201:
        print("  Created 'client_id' claim on Okta auth server")
    elif resp.status_code == 409:
        print("  'client_id' claim already exists")
    else:
        print(f"  Warning: Could not create claim ({resp.status_code}): {resp.text[:200]}")
else:
    print("  Skipping (no OKTA_API_TOKEN set) — ensure 'client_id' claim exists manually")

# --- Step 1: Create the Gateway with Okta CUSTOM_JWT authorizer ---
# allowedAudience uses "api://default" — Okta's default authorization server audience.
# allowedClients is seeded with the SPA client ID from 00_okta_setup.ipynb.
# Demo notebooks (02, 05) append their own client IDs later.
if not OKTA_SPA_CLIENT_ID:
    raise ValueError("OKTA_SPA_CLIENT_ID not set — run 00_okta_setup.ipynb first to create the SPA app")

initial_allowed_clients = [OKTA_SPA_CLIENT_ID]
print(f"\nCreating AgentCore Gateway with Okta OIDC authorizer...")
print(f"  Initial allowedClients: {initial_allowed_clients}")

try:
    gateway_response = agentcore_client.create_gateway(
        name=GATEWAY_NAME,
        description="Open Insurance MCP Gateway demo with Okta OIDC auth",
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": initial_allowed_clients,
            }
        },
        exceptionLevel="DEBUG",
    )
    GATEWAY_ID = gateway_response["gatewayId"]
    GATEWAY_URL = gateway_response.get("gatewayUrl", "")
    print(f"Created Gateway: {GATEWAY_ID}")
except agentcore_client.exceptions.ConflictException:
    # Gateway already exists — retrieve it
    gateways = agentcore_client.list_gateways()
    for gw in gateways.get("items", []):
        if gw["name"] == GATEWAY_NAME:
            GATEWAY_ID = gw["gatewayId"]
            break

# Wait for Gateway to be READY
print("Waiting for Gateway to be READY...")
for attempt in range(30):
    gw = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    status = gw["status"]
    if status == "READY":
        GATEWAY_URL = gw["gatewayUrl"]
        print(f"Gateway READY: {GATEWAY_URL}")
        break
    elif status == "FAILED":
        print(f"Gateway FAILED: {gw}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")
else:
    print("Warning: Gateway did not become READY in time")

# --- Credential provider config (Gateway IAM role invokes Lambda) ---
LAMBDA_CRED_CONFIG = [{"credentialProviderType": "GATEWAY_IAM_ROLE"}]

# --- Step 2: Add PolicyLookup Lambda as a target (unrestricted) ---
print("\nAdding targets to Gateway...")

target_ids = {}

try:
    resp = agentcore_client.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name="PolicyLookup",
        description="Insurance policy lookup — accessible by all authenticated users",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": POLICY_LOOKUP_LAMBDA_ARN,
                    "toolSchema": {
                        "inlinePayload": [{
                            "name": "lookup_policy",
                            "description": "Look up insurance policy details by policy number. Available policies: POL-10042, POL-10078, POL-10103, POL-10156, POL-10201.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "policy_number": {"type": "string", "description": "Policy number (e.g., POL-10042)"}
                                },
                                "required": ["policy_number"]
                            }
                        }]
                    }
                }
            }
        },
        credentialProviderConfigurations=LAMBDA_CRED_CONFIG,
    )
    target_ids["PolicyLookup"] = resp["targetId"]
    print(f"  Added target: PolicyLookup (Lambda) [{resp['targetId']}]")
except agentcore_client.exceptions.ConflictException:
    print("  PolicyLookup target already exists")

# --- Step 3: Add ClaimsData Lambda as a target (restricted + contextual) ---
try:
    resp = agentcore_client.create_gateway_target(
        gatewayIdentifier=GATEWAY_ID,
        name="ClaimsData",
        description="Insurance claims data with optional amount filtering. Contains CONFIDENTIAL claims.",
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": CLAIMS_LAMBDA_ARN,
                    "toolSchema": {
                        "inlinePayload": [{
                            "name": "query_claims",
                            "description": "Query insurance claims data. Contains CONFIDENTIAL claims information including adjuster notes. Search by claim ID, policy number, holder name, or ask for 'claims summary'. Use max_amount to filter by claim size.",
                            "inputSchema": {
                                "type": "object",
                                "properties": {
                                    "query": {"type": "string", "description": "Claims query — e.g., 'claims summary', 'CLM-2024-0891', 'POL-10042', or a holder name"},
                                    "max_amount": {"type": "integer", "description": "Maximum claim amount to return (e.g., 100000 for claims <= $100K)"}
                                },
                                "required": ["query"]
                            }
                        }]
                    }
                }
            }
        },
        credentialProviderConfigurations=LAMBDA_CRED_CONFIG,
    )
    target_ids["ClaimsData"] = resp["targetId"]
    print(f"  Added target: ClaimsData (Lambda) [{resp['targetId']}]")
except agentcore_client.exceptions.ConflictException:
    print("  ClaimsData target already exists")

# --- Backfill target_ids from existing targets if any were skipped ---
if len(target_ids) < 2:
    existing_targets = agentcore_client.list_gateway_targets(gatewayIdentifier=GATEWAY_ID)
    for t in existing_targets.get("items", []):
        if t["name"] not in target_ids:
            target_ids[t["name"]] = t["targetId"]
            print(f"  Retrieved existing target: {t['name']} [{t['targetId']}]")

print(f"\n  Gateway ready: {GATEWAY_URL}")
print(f"  Gateway ID: {GATEWAY_ID}")
print(f"  Targets: {list(target_ids.keys())}")

## Cell 5: Define Cedar Policies (ENFORCE mode)

Cedar policies define **who can access what** through the Gateway:
- `PolicyLookup` — accessible by **all authenticated OAuth users** (customer service + claims adjusters)
- `ClaimsData` — **full access** only for Bob (claims adjuster)
- `ClaimsData` — **contextual access** for the `CONTEXTUAL_USER`: only when `max_amount <= CONTEXTUAL_MAX_AMOUNT`

### Cedar Entity Model for AgentCore Gateway

| Entity | Format | Example |
|--------|--------|---------|
| **Principal** | `AgentCore::OAuthUser::"<JWT sub>"` | `AgentCore::OAuthUser::"bob@example.com"` |
| **Action** | `AgentCore::Action::"<TargetName>"` | `AgentCore::Action::"PolicyLookup"` |
| **Action (tool)** | `AgentCore::Action::"<Target>___<tool>"` | `AgentCore::Action::"PolicyLookup___lookup_policy"` |
| **Resource** | `AgentCore::Gateway::"<gateway-arn>"` | `AgentCore::Gateway::"arn:aws:..."` |

### Contextual Control with `when` clause

Cedar's `when` clause can constrain **tool input values** via `context.input`:
```cedar
permit(principal, action, resource)
when { context.input has "max_amount" && context.input.max_amount <= 100000 };
```

This means the Gateway will **deny** the tool call if:
- `max_amount` is not provided (field missing)
- `max_amount > 100000` (exceeds limit)

**Important:** Use `context.input` (not `context.toolInput`) for tool-level actions (`Target___tool`).
Target-level actions use `context.system` instead.

**Note:** The Gateway maps JWT `sub` claim to `AgentCore::OAuthUser` principal entities.
Group-based attributes (e.g., `principal.groups`) are not currently available in the Cedar schema —
use explicit principal matching for fine-grained access control.

In [ ]:
# --- Ensure Gateway role has policy engine permissions ---
iam_client = boto3.client("iam")
import json as json_mod

pe_policy_doc = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": "bedrock-agentcore:*",
        "Resource": "*"
    }]
}

iam_client.put_role_policy(
    RoleName=f"{DEMO_PREFIX}-gateway-role",
    PolicyName="policy-engine-access",
    PolicyDocument=json_mod.dumps(pe_policy_doc),
)
print("Added bedrock-agentcore permissions to Gateway role")

# --- Create Policy Engine for the Gateway ---
print("\nCreating Policy Engine for access control...\n")

POLICY_ENGINE_NAME = "agentcore_mcp_demo_policy_engine"

try:
    pe_response = agentcore_client.create_policy_engine(
        name=POLICY_ENGINE_NAME,
        description="Cedar policy engine for Open Insurance Gateway demo (ENFORCE mode)",
    )
    POLICY_ENGINE_ID = pe_response["policyEngineId"]
    print(f"Created Policy Engine: {POLICY_ENGINE_ID}")
except agentcore_client.exceptions.ConflictException:
    # Already exists — find it
    engines = agentcore_client.list_policy_engines()
    for pe in engines.get("policyEngines", engines.get("items", [])):
        if pe["name"] == POLICY_ENGINE_NAME:
            POLICY_ENGINE_ID = pe["policyEngineId"]
            print(f"Policy Engine already exists: {POLICY_ENGINE_ID}")
            break

# Wait for Policy Engine to be ACTIVE
print("Waiting for Policy Engine to be ACTIVE...")
for attempt in range(30):
    pe = agentcore_client.get_policy_engine(policyEngineId=POLICY_ENGINE_ID)
    status = pe["status"]
    if status == "ACTIVE":
        print(f"Policy Engine ACTIVE")
        break
    elif status == "FAILED":
        print(f"Policy Engine FAILED: {pe}")
        break
    time.sleep(5)
    print(f"  Status: {status} ({(attempt + 1) * 5}s)")

# --- Attach Policy Engine to Gateway in ENFORCE mode ---
# Read current allowedClients to avoid overwriting clients registered by demo notebooks
print("\nAttaching Policy Engine to Gateway (ENFORCE mode)...")
time.sleep(10)  # Allow IAM propagation

gw = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
current_clients = gw["authorizerConfiguration"]["customJWTAuthorizer"]["allowedClients"]

try:
    agentcore_client.update_gateway(
        gatewayIdentifier=GATEWAY_ID,
        name=GATEWAY_NAME,
        roleArn=GATEWAY_ROLE_ARN,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": OKTA_DISCOVERY_URL,
                "allowedAudience": ["api://default"],
                "allowedClients": current_clients,
            }
        },
        policyEngineConfiguration={
            "arn": pe["policyEngineArn"],
            "mode": "ENFORCE",
        },
        exceptionLevel="DEBUG",
    )
    print("Policy Engine attached to Gateway (ENFORCE mode)")
except Exception as e:
    print(f"Warning: Could not attach policy engine: {e}")

# Wait for Gateway to be READY again after update
print("Waiting for Gateway to settle after policy engine attachment...")
for attempt in range(30):
    gw = agentcore_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
    if gw["status"] == "READY":
        print("Gateway READY")
        break
    time.sleep(5)
    print(f"  Status: {gw['status']} ({(attempt + 1) * 5}s)")

# --- Define Cedar policies ---
GATEWAY_ARN = gw["gatewayArn"]

# Policy 1: All authenticated OAuthUsers can access PolicyLookup
POLICY_LOOKUP_POLICY = f"""permit(
    principal is AgentCore::OAuthUser,
    action in [AgentCore::Action::"PolicyLookup", AgentCore::Action::"PolicyLookup___lookup_policy"],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
);"""

# Policy 2: Only Bob (claims adjuster) can access ClaimsData — full access, no constraints
CLAIMS_POLICY = f"""permit(
    principal == AgentCore::OAuthUser::"{BOB_USERNAME}",
    action in [AgentCore::Action::"ClaimsData", AgentCore::Action::"ClaimsData___query_claims"],
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
);"""

policies = {
    "allow_policy_lookup_all": POLICY_LOOKUP_POLICY,
    "allow_claims_bob": CLAIMS_POLICY,
}

# Policy 3: Contextual demo user — ClaimsData access ONLY when max_amount <= threshold
if CONTEXTUAL_USER:
    CLAIMS_CONTEXTUAL = f"""permit(
    principal == AgentCore::OAuthUser::"{CONTEXTUAL_USER}",
    action == AgentCore::Action::"ClaimsData___query_claims",
    resource == AgentCore::Gateway::"{GATEWAY_ARN}"
)
when {{ context.input has "max_amount" && context.input.max_amount <= {CONTEXTUAL_MAX_AMOUNT} }};"""
    policies["allow_claims_contextual"] = CLAIMS_CONTEXTUAL
else:
    print("\n⚠️  CONTEXTUAL_USER not set in .env — skipping contextual Cedar policy")
    print("   Set CONTEXTUAL_USER to enable the Cedar contextual control demo")

# --- Detect and clean up stale policies referencing a different gateway ARN ---
existing_policies = agentcore_client.list_policies(policyEngineId=POLICY_ENGINE_ID)
existing_by_name = {}
stale_policies = []
for p in existing_policies.get("policies", existing_policies.get("items", [])):
    existing_by_name[p["name"]] = p
    statement = p.get("definition", {}).get("cedar", {}).get("statement", "")
    if GATEWAY_ARN not in statement:
        stale_policies.append(p)

if stale_policies:
    print(f"\n⚠️  Found {len(stale_policies)} stale policy(ies) referencing a different gateway — cleaning up...")
    for sp in stale_policies:
        try:
            agentcore_client.delete_policy(policyEngineId=POLICY_ENGINE_ID, policyId=sp["policyId"])
            print(f"  Deleted stale policy: {sp['name']} ({sp['policyId']})")
            existing_by_name.pop(sp["name"], None)
        except Exception as e:
            print(f"  Warning: could not delete {sp['name']}: {e}")
    time.sleep(15)

# Create policies on the Policy Engine (skip if already exists with correct gateway ARN)
for policy_name, policy_body in policies.items():
    if policy_name in existing_by_name:
        print(f"  Policy '{policy_name}' already exists (gateway ARN matches)")
        continue
    try:
        resp = agentcore_client.create_policy(
            name=policy_name,
            policyEngineId=POLICY_ENGINE_ID,
            definition={
                "cedar": {
                    "statement": policy_body.strip()
                }
            },
        )
        print(f"  Policy '{policy_name}' created")
    except agentcore_client.exceptions.ConflictException:
        print(f"  Policy '{policy_name}' already exists")
    except Exception as e:
        print(f"  Warning: Policy '{policy_name}': {e}")

# Verify policies are ACTIVE
time.sleep(10)
print("\nVerifying policy statuses...")
all_active = True
policy_list = agentcore_client.list_policies(policyEngineId=POLICY_ENGINE_ID)
for p in policy_list.get("policies", policy_list.get("items", [])):
    status_icon = "✅" if p["status"] == "ACTIVE" else "❌"
    print(f"  {status_icon} {p['name']}: {p['status']}")
    if p.get("statusReasons"):
        for r in p["statusReasons"]:
            print(f"      {r}")
    if p["status"] != "ACTIVE":
        all_active = False

print("\n--- Policy Summary (Open Insurance) ---")
print(f"PolicyLookup:  ALL authenticated users           -> ALLOW")
print(f"ClaimsData:    {BOB_USERNAME} (claims adjuster)  -> ALLOW (full access)")
if CONTEXTUAL_USER:
    print(f"ClaimsData:    {CONTEXTUAL_USER} (contextual)       -> ALLOW only when max_amount <= {CONTEXTUAL_MAX_AMOUNT}")
print(f"ClaimsData:    all others (e.g., Alice)          -> DENY (no matching permit)")
print(f"\nMode: ENFORCE (policies enforced by Gateway)")
if all_active:
    print("✅ All policies ACTIVE — Gateway-level access control is working")
else:
    print("⚠️  Some policies failed — check status reasons above")

## Cell 6: Save Configuration

Saves all resource identifiers to `gateway_config.json` for the demo notebooks.

In [ ]:
config = {
    "gateway_id": GATEWAY_ID,
    "gateway_url": GATEWAY_URL,
    "gateway_arn": GATEWAY_ARN,
    "okta_domain": OKTA_DOMAIN,
    "okta_issuer": OKTA_ISSUER,
    "aws_region": AWS_REGION,
    "policy_engine_id": POLICY_ENGINE_ID,
    "policy_engine_arn": pe["policyEngineArn"],
    "targets": {
        "policy_lookup": {"name": "PolicyLookup", "type": "lambda", "arn": POLICY_LOOKUP_LAMBDA_ARN,
                     "target_id": target_ids.get("PolicyLookup", "")},
        "claims": {"name": "ClaimsData", "type": "lambda", "arn": CLAIMS_LAMBDA_ARN,
                     "target_id": target_ids.get("ClaimsData", "")},
    },
    "demo_resources": {
        "lambda_role": LAMBDA_ROLE_NAME,
        "gateway_role": GATEWAY_ROLE_ARN,
        "policy_lookup_lambda": POLICY_LOOKUP_FUNCTION_NAME,
        "claims_lambda": CLAIMS_FUNCTION_NAME,
    }
}

config_path = os.path.join(os.getcwd(), "gateway_config.json")
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

print(f"✅ Config saved to: {config_path}")
print(f"\nConfiguration:")
print(json.dumps(config, indent=2))
print(f"\n🎉 Setup complete! Run 02_claude_code_setup.ipynb or 05_strands_demo.ipynb to demo.")

## Cleanup (run after demo is complete)

Run this cell to delete all AWS resources created by this notebook.

- **Terraform destroy** removes all infrastructure: Lambda functions, IAM roles
- **AgentCore API** deletes the Gateway, targets, Policy Engine, and Cedar policies

> **Note:** Demo users (Alice, Bob) are managed by `00_okta_setup.ipynb` — run that notebook's cleanup to remove them from Okta.

In [ ]:
# ⚠️  CLEANUP — This deletes all demo resources. Only run when you're done!

import json
import subprocess
import time
import boto3
from dotenv import load_dotenv

load_dotenv(override=True)

with open("gateway_config.json") as f:
    cfg = json.load(f)

region = cfg["aws_region"]
agentcore = boto3.client("bedrock-agentcore-control", region_name=region)

gw_id = cfg["gateway_id"]
pe_id = cfg.get("policy_engine_id", "")

print("Cleaning up demo resources...\n")

# 1. Delete all Cedar policies from the policy engine (must happen before policy engine deletion)
if pe_id:
    try:
        policy_list = agentcore.list_policies(policyEngineId=pe_id)
        policies = policy_list.get("policies", policy_list.get("items", []))
        if policies:
            print(f"  Deleting {len(policies)} Cedar policy(ies)...")
            for p in policies:
                try:
                    agentcore.delete_policy(policyEngineId=pe_id, policyId=p["policyId"])
                    print(f"    Deleted policy: {p['name']} ({p['policyId']})")
                except Exception as e:
                    print(f"    Skip policy {p['name']}: {e}")
            print("  Waiting for policy deletion to propagate...")
            time.sleep(15)
        else:
            print("  No policies to delete")
    except Exception as e:
        print(f"  Skip policy cleanup: {e}")

# 2. Delete Policy Engine (must happen after policies are deleted)
if pe_id:
    try:
        agentcore.delete_policy_engine(policyEngineId=pe_id)
        print(f"  Deleted Policy Engine: {pe_id}")
        time.sleep(10)
    except Exception as e:
        print(f"  Skip Policy Engine: {e}")

# 3. Delete Gateway targets (must happen before gateway deletion)
try:
    existing_targets = agentcore.list_gateway_targets(gatewayIdentifier=gw_id)
    for t in existing_targets.get("items", []):
        try:
            agentcore.delete_gateway_target(gatewayIdentifier=gw_id, targetId=t["targetId"])
            print(f"  Deleted target: {t['name']} ({t['targetId']})")
        except Exception as e:
            print(f"  Skip target {t['name']}: {e}")
except Exception as e:
    print(f"  Skip target cleanup: {e}")

# Also try targets from config (in case list_gateway_targets missed any)
for target_key, target_info in cfg.get("targets", {}).items():
    tid = target_info.get("target_id", "")
    if tid:
        try:
            agentcore.delete_gateway_target(gatewayIdentifier=gw_id, targetId=tid)
            print(f"  Deleted target: {target_info['name']} ({tid})")
        except Exception as e:
            pass  # Already deleted above or doesn't exist

# 4. Delete Gateway (must happen after all targets are deleted)
print("  Waiting for target deletions to propagate...")
time.sleep(10)
try:
    agentcore.delete_gateway(gatewayIdentifier=gw_id)
    print(f"  Deleted Gateway: {gw_id}")
except Exception as e:
    print(f"  Skip Gateway: {e}")

# 5. Destroy all infrastructure via Terraform (Lambda, IAM)
TERRAFORM_DIR = os.path.join(os.getcwd(), "terraform")
DEMO_PREFIX = "agentcore-mcp-demo"
print("\n  Running terraform destroy...")
try:
    subprocess.run(
        ["terraform", "destroy", "-auto-approve",
         f"-var=aws_region={region}", f"-var=demo_prefix={DEMO_PREFIX}",
         "-var=contextual_user=", "-var=contextual_max_amount=100000"],
        cwd=TERRAFORM_DIR, check=True,
    )
    print("  ✅ Terraform destroy complete")
except Exception as e:
    print(f"  ⚠️  Terraform destroy failed: {e}")

print("\n✅ Cleanup complete!")
print("Note: Demo users (Alice, Bob) are managed by 00_okta_setup.ipynb — run its cleanup to remove them.")